# Binary Classification - PyTorch

Develop an End-to-End training with PyTorch best practices for a binary classification model.

# Notebook Setup

## Imports

In [1]:
# Import Standard Libraries
import numpy as np
import os
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, dataloader
from torch.utils.tensorboard import SummaryWriter

## Define Configurations

In [2]:
# Define root path
root_path = Path(os.getcwd()).parents[2]

# Data path
data_path = root_path / 'data' / 'diabetes.csv'

# TensorBoard
tb_writer = SummaryWriter('runs/binary_classification')

# DataLoader configurations
batch_size = 42
shuffle = True

# Read Data

In [3]:
# Read data
data = pd.read_csv(data_path)

In [4]:
data

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


# Data Preparation

## Train & Test Split

In [6]:
# Define features and label
x = data.drop(columns=['Outcome'], axis=1).values
y = data.Outcome.values

In [7]:
# Split between train and validation set
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## Standardisation

In [8]:
# Standardise the data
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## Dataset

In [9]:
class DiabetesDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Instance dataset
train_dataset = DiabetesDataset(x_train_scaled, y_train)
test_dataset = DiabetesDataset(x_test_scaled, y_test)

## DataLoader

In [10]:
# Instance dataloader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=shuffle)

## Log Data Sample

In [11]:
# Log features and a data sample
tb_writer.add_text('Feature: ', ', '.join(data.columns), global_step=0)
tb_writer.add_text("Sample Data", f"```\n{data.sample(1).to_markdown(index=False)}\n```", global_step=0)
tb_writer.flush()

# Model Definition

## Select Accelerator

In [12]:
# Define the device to work on
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


## Define Architecture

In [13]:
# Define the model
class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(8, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, network_input):
        return self.layers(network_input)

In [14]:
# Instance model
model = DiabetesNet().to(device)
print(model)

DiabetesNet(
  (layers): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Linear(in_features=16, out_features=1, bias=True)
    (8): Sigmoid()
  )
)


# Model Training

## Loss & Optimiser

In [15]:
# Define loss and optimiser
loss_function = nn.BCELoss()
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

## Train & Test Functions

In [100]:
def train_model (dataloader, model, loss_function, optimiser):
    """Train the model and return the loss"""

    # Initialise loss
    loss = None

    # Compute number of batches
    batches = len(dataloader)

    # Switch model to training mode
    model.train()

    # Fetch the batches
    for batch, (X, y) in enumerate(dataloader):

        # Load data into device
        X, y = X.to(device), y.to(device)

        # Compute the loss
        predictions = model(X)
        loss = loss_function(predictions, y)

        # Backpropagation
        loss.backward() # Compute gradients
        optimiser.step() # Update weights
        optimiser.zero_grad() # Clear gradients buffer

        # Logging every quarter of batches
        if batch + 1 in np.linspace(1, len(dataloader), 4, dtype=int):

            print(predictions.data.cpu().numpy().argmax())
            
            # Compute loss and accuracy
            loss_value, accuracy = loss.item(), accuracy_score(y, predictions.cpu().detach().numpy())

            #print(f"loss: {loss_value:>7f}  [{current_batch:>5d}/{size:>5d}]")
            print(f'Loss: {loss_value:>5f} - Accuracy: {accuracy:>5f} - [{batch + 1}/{batches}]')
            #Log on TensorBoard

    return loss.item()

In [101]:
def test_model (dataloader, model, loss_function):
    """Test the model"""


    size = len(dataloader.dataset)

In [102]:
train_model(train_loader, model, loss_function, optimiser)

35
tensor(35, device='mps:0')


TypeError: can't convert mps:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [ ]:
15/100 = 5/x -> x = 5 * 100/15